Download dataset

In [2]:
#
# clean main folders
!rm -r *
!ls

# Creating a dataset directory.
# -p Will also create any intermediate directories that don't exist
!mkdir -p ./datasets/us-baby-name/zip
!mkdir -p ./datasets/us-baby-name/data

# download the dataset from SSA and unzip it
!wget 'https://www.ssa.gov/oact/babynames/state/namesbystate.zip' -O "./datasets/us-baby-name/zip/namesbystate.zip"
!unzip {"./datasets/us-baby-name/zip/namesbystate.zip"} -d {"./datasets/us-baby-name/data"}

--2026-03-03 02:34:48--  https://www.ssa.gov/oact/babynames/state/namesbystate.zip
Resolving www.ssa.gov (www.ssa.gov)... 23.211.124.209, 23.211.124.208, 2600:1407:3c00:cc::172d:2ec9, ...
Connecting to www.ssa.gov (www.ssa.gov)|23.211.124.209|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27485972 (26M) [application/zip]
Saving to: ‘./datasets/us-baby-name/zip/namesbystate.zip’

./datasets/us-baby- 100%[===================>]  26.21M  47.1MB/s    in 0.6s    

2026-03-03 02:34:49 (47.1 MB/s) - ‘./datasets/us-baby-name/zip/namesbystate.zip’ saved [27485972/27485972]

Archive:  ./datasets/us-baby-name/zip/namesbystate.zip
  inflating: ./datasets/us-baby-name/data/NE.TXT  
  inflating: ./datasets/us-baby-name/data/NH.TXT  
  inflating: ./datasets/us-baby-name/data/NJ.TXT  
  inflating: ./datasets/us-baby-name/data/NM.TXT  
  inflating: ./datasets/us-baby-name/data/NV.TXT  
  inflating: ./datasets/us-baby-name/data/NY.TXT  
  inflating: ./datasets/us-baby-name/data

In [3]:
# Test the downloaded data
!ls 'datasets/us-baby-name/data'

AK.TXT	CT.TXT	IA.TXT	LA.TXT	MO.TXT	NH.TXT	OK.TXT	StateReadMe.pdf  WA.TXT
AL.TXT	DC.TXT	ID.TXT	MA.TXT	MS.TXT	NJ.TXT	OR.TXT	TN.TXT		 WI.TXT
AR.TXT	DE.TXT	IL.TXT	MD.TXT	MT.TXT	NM.TXT	PA.TXT	TX.TXT		 WV.TXT
AZ.TXT	FL.TXT	IN.TXT	ME.TXT	NC.TXT	NV.TXT	RI.TXT	UT.TXT		 WY.TXT
CA.TXT	GA.TXT	KS.TXT	MI.TXT	ND.TXT	NY.TXT	SC.TXT	VA.TXT
CO.TXT	HI.TXT	KY.TXT	MN.TXT	NE.TXT	OH.TXT	SD.TXT	VT.TXT


Create dataframe

In [4]:
import glob
import pandas as pd

path = 'datasets/us-baby-name/data'
file_list = glob.glob(f'{path}/*.TXT')
all_name_df = None
for file_path in file_list:
  df_file = pd.read_csv(file_path, sep=",", header=None)
  df_file.columns = ['State', 'Gender',  'Year', 'Name', 'Number']
  if all_name_df is None:
    all_name_df = df_file
  else:
    all_name_df = pd.concat([all_name_df, df_file])
all_name_df = all_name_df[['State', 'Gender', 'Name', 'Number', 'Year']]  # rearrange the columns
print(all_name_df.head(5))

  State Gender      Name  Number  Year
0    OH      F      Mary    1099  1910
1    OH      F     Helen     698  1910
2    OH      F   Dorothy     487  1910
3    OH      F      Ruth     457  1910
4    OH      F  Margaret     452  1910


In [5]:
# Check all_name_df's dimension (x, y)
all_name_df.shape

(6600640, 5)

Create SQL DB using the dataframe

In [6]:
!mkdir -p ./datasets/us-baby-name/sql

import sqlite3
DB_PATH_BABY = 'datasets/us-baby-name/sql/database.sqlite'
!rm -f ./datasets/us-baby-name/sql/database.sqlite

to_db = list(all_name_df.itertuples(index=False))
con = sqlite3.connect(DB_PATH_BABY)
cur = con.cursor()
# creates a table named (Names) with the dataset data and the following columns:
# 'State', 'Gender', 'Name', 'Number', and 'Year' (5pt)
cur.execute("CREATE TABLE babyName (State, Gender, Name, Number, Year);")

#  Load the data using a Batch INSERT SQL Query (2pt):
cur.executemany("INSERT INTO babyName (State, Gender, Name, Number, Year) VALUES (?, ?, ?, ?, ?);", to_db)
con.commit()
con.close()

In [7]:
# Test the database

!ls datasets/us-baby-name/sql/
conn = sqlite3.connect(DB_PATH_BABY) # connecting to the database
c = conn.cursor() # creating a cursor object
print("The number of names in the dataset: %s" % c.execute("SELECT count(*) FROM babyName" ).fetchone()) # execute a query & fetch the results
c.close() # close the cursor

database.sqlite
The number of names in the dataset: 6600640


In [8]:
import timeit
import sqlite3

!ls datasets/us-baby-name/sql/

# Connect to DB
conn = sqlite3.connect(DB_PATH_BABY)
# Create a cursor object
c = conn.cursor()
# Write a query that returns the statistics for the name William (5pt)
# Use the the timeit package to measure the time it takes the query to run (5pt).
query_for_stat = "SELECT Gender, sum(Number) FROM babyName WHERE Name == 'William' GROUP BY Gender"
print(f"The number of Female (F) and Male (M) named Williams in the dataset: {c.execute(query_for_stat).fetchall()}")
conn.commit()
conn.close() # close the cursor

# Use the the timeit package to measure the time it takes the query to run (5pt).
num_of_runs = 100
without_time = timeit.timeit(f'c.execute("{query_for_stat}").fetchone();', setup='import sqlite3; conn = sqlite3.connect(\'datasets/us-baby-name/sql/database.sqlite\'); c = conn.cursor();', number=num_of_runs)
print(f'Time to run query without indices (average of {num_of_runs} runs): {without_time/num_of_runs:.5f}')

# Create an index on the Name column and use the the timeit package to measure the time it takes the query to run with the index (5pt)
conn = sqlite3.connect(DB_PATH_BABY) # connecting to the database
c = conn.cursor() # creating a cursor object
# c.execute("DROP INDEX name_idx;")
c.execute("CREATE INDEX name_idx ON babyName (Name)")
conn.commit()
conn.close() # close the cursor

with_time = timeit.timeit(f'c.execute("{query_for_stat}").fetchone();', setup='import sqlite3; conn = sqlite3.connect(\'datasets/us-baby-name/sql/database.sqlite\'); c = conn.cursor();', number=num_of_runs)
print(f'Time to run query with indices (average of {num_of_runs} runs): {with_time/num_of_runs:.5f}')

database.sqlite
The number of Female (F) and Male (M) named Williams in the dataset: [('F', 10212), ('M', 3962559)]
Time to run query without indices (average of 100 runs): 0.38141
Time to run query with indices (average of 100 runs): 0.01391


Create a NoSQL database on TinyDB using the dataframe

In [ ]:
!pip install tinydb
from tinydb import TinyDB

df_records = all_name_df.to_dict(orient='records')
print(f"DataFrame converted to records: {df_records}")

KeyboardInterrupt: 

In [ ]:
# 3. Initialize TinyDB and insert the records
try:
    # Creates/opens a JSON file named 'dataframe_db.json'
    db = TinyDB('dataframe_db.json')
    # Optional: Clear the table before inserting
    db.truncate()

    # Insert the list of records into the database
    db.insert_multiple(df_records)
    print("\nData successfully inserted into TinyDB.")

    # 4. (Optional) Verify the contents
    print("\nContents of the database:")
    for item in db.all():
        print(item)

finally:
    # Ensure the database connection is closed
    db.close()

Buffered data was truncated after reaching the output size limit.

Big data approach: chunking or paginating

In [ ]:
# Paging parameters
page_size = 100000
total_rows = len(all_name_df)
num_pages = (total_rows + page_size - 1) // page_size

try:
  # Creates/opens a JSON file named 'dataframe_db.json'
  db = TinyDB('dataframe_db.json')
  # Optional: Clear the table before inserting
  db.truncate()

  # 3. Loop through pages
  for i in range(0, total_rows, page_size):
      # Slice the DataFrame for the current page
      page_df = all_name_df.iloc[i:i + page_size]
      print(f"--- Processing Page {i//page_size + 1} / {num_pages}")

      # 4. Loop through rows in the page
      #for index, row in page_df.iterrows():
        #print(f"Index: {index}, Name: {row['Name']}, Value: {row['Year']}")
      df_records = page_df.to_dict(orient='records')
      #print(f"DataFrame converted to records: {df_records}")

      # Insert the list of records into the database
      db.insert_multiple(df_records)
      #print("\nData successfully inserted into TinyDB.")


  # 4. (Optional) Verify the contents
  print("\nContents of the database:")
  for item in db.all():
      print(item)
finally:
    # Ensure the database connection is closed
    db.close()

Streaming output truncated to the last 5000 lines.
{'State': 'IN', 'Gender': 'M', 'Name': 'Jaden', 'Number': 19, 'Year': 2020}
{'State': 'IN', 'Gender': 'M', 'Name': 'Jaiden', 'Number': 19, 'Year': 2020}
{'State': 'IN', 'Gender': 'M', 'Name': 'Jamison', 'Number': 19, 'Year': 2020}
{'State': 'IN', 'Gender': 'M', 'Name': 'Jase', 'Number': 19, 'Year': 2020}
{'State': 'IN', 'Gender': 'M', 'Name': 'Kendrick', 'Number': 19, 'Year': 2020}
{'State': 'IN', 'Gender': 'M', 'Name': 'Ledger', 'Number': 19, 'Year': 2020}
{'State': 'IN', 'Gender': 'M', 'Name': 'Phillip', 'Number': 19, 'Year': 2020}
{'State': 'IN', 'Gender': 'M', 'Name': 'Quinton', 'Number': 19, 'Year': 2020}
{'State': 'IN', 'Gender': 'M', 'Name': 'Raiden', 'Number': 19, 'Year': 2020}
{'State': 'IN', 'Gender': 'M', 'Name': 'Rhys', 'Number': 19, 'Year': 2020}
{'State': 'IN', 'Gender': 'M', 'Name': 'Ronin', 'Number': 19, 'Year': 2020}
{'State': 'IN', 'Gender': 'M', 'Name': 'Seth', 'Number': 19, 'Year': 2020}
{'State': 'IN', 'Gender': 'M

In [ ]:
# Assignment
# Cari nilai page_size terbaik untuk mendapatkan waktu insertion terpendek untuk RAM yang tersedia
# dengan membuat grafik plot waktu (y-axis) terhadap page_size (x-axis)
# Ambil rata-rata waktu insertion per row, bandingkan dengan waktu insertion di SQL
# Jelaskan hasil temuanmu.